# Problem 6: Ride Fare Estimation with Google TabFM

## Problem
A ride-hailing company must estimate trip fare **before the ride starts**.

## What to do
Predict expected fare amount from trip details.

## Goals
- Display accurate fare estimates
- Improve customer trust
- Optimize pricing strategies

Target: `fare` (continuous)

This notebook is production-oriented: direct web download, leakage-aware temporal split, historical context features, baseline+TabFM modeling, and deployable artifacts.

## 0) Why this design

### Dataset
We use the NYC taxi dataset published via seaborn repository and downloaded directly from web:
`https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv`

This dataset includes:
- pickup/destination info (`pickup_zone`, `dropoff_zone`, boroughs)
- trip distance (`distance`)
- timestamps (`pickup`, `dropoff`) for travel-time proxies
- fare amount (`fare`) target

### Pre-trip realism
To emulate pre-trip estimation, we **do not use post-trip fields** like `tip`, `tolls`, `total`, `payment`.
We build train-history-based features:
- estimated travel time by route-hour profile
- traffic pace profile by pickup borough-hour
- demand level by pickup zone-hour

### Modeling strategy
- Baseline: XGBoost regressor
- Foundation model: Google TabFM regressor (3 variants)
- Evaluation: RMSE/MAE/R2/MAPE + uncertainty tiers

## 1) Reproducible setup

Optional environment variables:

- `TABFM_DEVICE=auto|cpu|cuda`
- `RIDE_DATA_URL=https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv`
- `RIDE_SAMPLE_TRAIN_ROWS=0` (0 means full)
- `RIDE_SAMPLE_EVAL_ROWS=0` (0 means full)
- `RIDE_MIN_SAMPLE_ROWS=200`
- `TABFM_CONTEXT_MAX_ROWS=1200`
- `TABFM_EVAL_MAX_ROWS=0` (0 means full val/test)
- `TABFM_FAST_MODE=0|1`
- `TABFM_CHECKPOINT_PATH=/abs/path/to/regression/model/or/repo/root` (optional)

License note:
- TabFM code is Apache-2.0.
- TabFM released weights are non-commercial; review terms before commercial use.

In [ ]:
from __future__ import annotations

import json
import os
import random
import time
import urllib.request
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import torch
from loguru import logger
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

from tabfm import TabFMRegressor
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1_0_0

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TABFM_DEVICE_PREF = os.getenv('TABFM_DEVICE', 'auto').lower().strip()
if TABFM_DEVICE_PREF not in {'auto', 'cpu', 'cuda'}:
    raise ValueError(f'Unsupported TABFM_DEVICE={TABFM_DEVICE_PREF}')

DATA_URL = os.getenv('RIDE_DATA_URL', 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv').strip()
SAMPLE_TRAIN_ROWS = int(os.getenv('RIDE_SAMPLE_TRAIN_ROWS', '0'))
SAMPLE_EVAL_ROWS = int(os.getenv('RIDE_SAMPLE_EVAL_ROWS', '0'))
MIN_SAMPLE_ROWS = int(os.getenv('RIDE_MIN_SAMPLE_ROWS', '200'))
TABFM_CONTEXT_MAX_ROWS = int(os.getenv('TABFM_CONTEXT_MAX_ROWS', '1200'))
TABFM_EVAL_MAX_ROWS = int(os.getenv('TABFM_EVAL_MAX_ROWS', '0'))
TABFM_FAST_MODE = os.getenv('TABFM_FAST_MODE', '0').strip() == '1'
TABFM_CHECKPOINT_OVERRIDE = os.getenv('TABFM_CHECKPOINT_PATH', '').strip()

if SAMPLE_TRAIN_ROWS != 0 and SAMPLE_TRAIN_ROWS <= MIN_SAMPLE_ROWS:
    raise ValueError(f'RIDE_SAMPLE_TRAIN_ROWS must be 0 or > {MIN_SAMPLE_ROWS}')
if SAMPLE_EVAL_ROWS != 0 and SAMPLE_EVAL_ROWS <= MIN_SAMPLE_ROWS:
    raise ValueError(f'RIDE_SAMPLE_EVAL_ROWS must be 0 or > {MIN_SAMPLE_ROWS}')
if TABFM_CONTEXT_MAX_ROWS <= 300:
    raise ValueError('TABFM_CONTEXT_MAX_ROWS must be > 300.')
if TABFM_EVAL_MAX_ROWS != 0 and TABFM_EVAL_MAX_ROWS <= 100:
    raise ValueError('TABFM_EVAL_MAX_ROWS must be 0 or > 100.')


def resolve_tabfm_device(preference: str) -> str:
    if preference == 'auto':
        return 'cuda' if torch.cuda.is_available() else 'cpu'
    if preference == 'cuda' and not torch.cuda.is_available():
        logger.warning('TABFM_DEVICE=cuda requested but CUDA unavailable; falling back to cpu')
        return 'cpu'
    return preference


def find_project_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / 'pyproject.toml').exists():
            return cand
    raise RuntimeError('Could not find project root (pyproject.toml not found).')


PROJECT_ROOT = find_project_root(Path.cwd())
PROBLEM_ROOT = PROJECT_ROOT / 'problems' / 'problem6_ride_fare_estimation'
DATA_DIR = PROBLEM_ROOT / 'data' / 'raw'
ARTIFACT_DIR = PROBLEM_ROOT / 'artifacts'

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = resolve_tabfm_device(TABFM_DEVICE_PREF)
TABFM_CKPT_PATH = Path(TABFM_CHECKPOINT_OVERRIDE) if TABFM_CHECKPOINT_OVERRIDE else None

logger.info('Project root: {}', PROJECT_ROOT)
logger.info('Problem root: {}', PROBLEM_ROOT)
logger.info('Raw data dir: {}', DATA_DIR)
logger.info('Artifacts dir: {}', ARTIFACT_DIR)
logger.info('TabFM device preference={} effective={}', TABFM_DEVICE_PREF, DEVICE)
logger.info('Sample train/eval rows: {}/{} (0 means full)', SAMPLE_TRAIN_ROWS, SAMPLE_EVAL_ROWS)
logger.info('TabFM context max rows: {}', TABFM_CONTEXT_MAX_ROWS)
logger.info('TabFM eval max rows: {} (0 means full)', TABFM_EVAL_MAX_ROWS)
logger.info('TabFM fast mode: {}', TABFM_FAST_MODE)
logger.info('TabFM checkpoint override: {}', TABFM_CKPT_PATH)

sns.set_theme(style='whitegrid')

## 2) Download data directly from web (idempotent)

In [ ]:
DATA_CSV = DATA_DIR / 'taxis.csv'

if DATA_CSV.exists() and DATA_CSV.stat().st_size > 0:
    logger.info('Using cached dataset {} ({:.1f} KB)', DATA_CSV, DATA_CSV.stat().st_size / 1024)
else:
    logger.info('Downloading dataset from {} ...', DATA_URL)
    urllib.request.urlretrieve(DATA_URL, DATA_CSV)
    logger.info('Downloaded to {} ({:.1f} KB)', DATA_CSV, DATA_CSV.stat().st_size / 1024)

DATA_CSV

## 3) Load and clean raw data

In [ ]:
raw_df = pd.read_csv(DATA_CSV)

required_cols = [
    'pickup', 'dropoff', 'passengers', 'distance', 'fare',
    'pickup_zone', 'dropoff_zone', 'pickup_borough', 'dropoff_borough',
]
missing = [c for c in required_cols if c not in raw_df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Keep only relevant columns for fare estimation and drop rows with missing essentials.
df = raw_df[required_cols].copy()

df['pickup'] = pd.to_datetime(df['pickup'], errors='coerce')
df['dropoff'] = pd.to_datetime(df['dropoff'], errors='coerce')

for c in ['passengers', 'distance', 'fare']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Filter invalid rows.
df = df.dropna(subset=['pickup', 'dropoff', 'distance', 'fare', 'pickup_zone', 'dropoff_zone', 'pickup_borough', 'dropoff_borough'])
df = df[df['fare'] > 0]
df = df[df['distance'] > 0]

df = df.reset_index(drop=True)
df.insert(0, 'trip_id', np.arange(len(df), dtype=np.int64))

logger.info('Clean shape: {}', df.shape)
logger.info('Fare summary: min=${:.2f} median=${:.2f} max=${:.2f}', df['fare'].min(), df['fare'].median(), df['fare'].max())

pd.DataFrame([
    {'metric': 'rows', 'value': len(df)},
    {'metric': 'fare_mean', 'value': float(df['fare'].mean())},
    {'metric': 'fare_median', 'value': float(df['fare'].median())},
    {'metric': 'distance_mean', 'value': float(df['distance'].mean())},
])

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['fare'], bins=40, kde=True)
plt.title('Fare Distribution')
plt.xlabel('Fare')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4) Time-aware split (train -> val -> test)

To mimic deployment and reduce leakage, we split chronologically by pickup timestamp.

In [ ]:
TARGET_COL = 'fare'
META_COLS = ['trip_id', 'fare', 'pickup', 'dropoff', 'distance']


df = df.sort_values('pickup').reset_index(drop=True)

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_full = df.iloc[:train_end].copy()
val_full = df.iloc[train_end:val_end].copy()
test_full = df.iloc[val_end:].copy()


def cap_time_rows(frame: pd.DataFrame, max_rows: int, keep: str = 'tail') -> pd.DataFrame:
    if max_rows == 0 or len(frame) <= max_rows:
        return frame.reset_index(drop=True)
    if keep == 'tail':
        return frame.tail(max_rows).reset_index(drop=True)
    return frame.head(max_rows).reset_index(drop=True)


train_df = cap_time_rows(train_full, SAMPLE_TRAIN_ROWS, keep='tail')
val_df = cap_time_rows(val_full, SAMPLE_EVAL_ROWS, keep='head')
test_df = cap_time_rows(test_full, SAMPLE_EVAL_ROWS, keep='head')

pd.DataFrame([
    {
        'split': 'train',
        'rows': len(train_df),
        'start': str(train_df['pickup'].min()),
        'end': str(train_df['pickup'].max()),
        'fare_mean': float(train_df['fare'].mean()),
    },
    {
        'split': 'val',
        'rows': len(val_df),
        'start': str(val_df['pickup'].min()),
        'end': str(val_df['pickup'].max()),
        'fare_mean': float(val_df['fare'].mean()),
    },
    {
        'split': 'test',
        'rows': len(test_df),
        'start': str(test_df['pickup'].min()),
        'end': str(test_df['pickup'].max()),
        'fare_mean': float(test_df['fare'].mean()),
    },
])

## 5) Leakage-aware feature engineering

Base features (available before ride starts):
- pickup/destination zones and boroughs
- pickup time context
- passenger count
- distance

Train-history features:
- estimated travel time by route-hour profile
- demand index by pickup zone-hour
- traffic pace profile by pickup borough-hour

All history maps are fit on **train only** and applied to val/test.

In [ ]:
def add_base_time_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()

    out['pickup_hour'] = out['pickup'].dt.hour
    out['pickup_dow'] = out['pickup'].dt.dayofweek
    out['pickup_month'] = out['pickup'].dt.month
    out['is_weekend'] = out['pickup_dow'].isin([5, 6]).astype(int)

    out['trip_duration_min'] = (out['dropoff'] - out['pickup']).dt.total_seconds() / 60.0
    out['trip_duration_min'] = out['trip_duration_min'].clip(lower=1.0)

    # Route key for historical profile joins.
    out['route_key'] = out['pickup_zone'].astype(str) + '->' + out['dropoff_zone'].astype(str)
    out['route_hour_key'] = out['route_key'] + '|h' + out['pickup_hour'].astype(str)

    return out


def fit_history_maps(train_frame: pd.DataFrame) -> dict[str, Any]:
    t = train_frame.copy()

    # Estimated travel time profile.
    route_hour_map = t.groupby('route_hour_key')['trip_duration_min'].median().to_dict()
    route_map = t.groupby('route_key')['trip_duration_min'].median().to_dict()
    global_trip_median = float(t['trip_duration_min'].median())

    # Demand profile by pickup zone-hour.
    demand_counts = t.groupby(['pickup_zone', 'pickup_hour']).size().rename('demand_count').reset_index()
    demand_map = {(r['pickup_zone'], int(r['pickup_hour'])): float(r['demand_count']) for _, r in demand_counts.iterrows()}
    demand_global = float(demand_counts['demand_count'].median()) if len(demand_counts) > 0 else 1.0
    demand_q1 = float(np.quantile(demand_counts['demand_count'], 0.33)) if len(demand_counts) > 0 else demand_global
    demand_q2 = float(np.quantile(demand_counts['demand_count'], 0.66)) if len(demand_counts) > 0 else demand_global

    # Traffic pace profile (minutes per mile) by pickup borough-hour.
    t['pace_min_per_mile'] = t['trip_duration_min'] / t['distance'].clip(lower=0.2)
    pace_stats = t.groupby(['pickup_borough', 'pickup_hour'])['pace_min_per_mile'].median().reset_index()
    pace_map = {(r['pickup_borough'], int(r['pickup_hour'])): float(r['pace_min_per_mile']) for _, r in pace_stats.iterrows()}
    pace_global = float(t['pace_min_per_mile'].median())
    pace_q1 = float(np.quantile(t['pace_min_per_mile'], 0.33))
    pace_q2 = float(np.quantile(t['pace_min_per_mile'], 0.66))

    return {
        'route_hour_map': route_hour_map,
        'route_map': route_map,
        'global_trip_median': global_trip_median,
        'demand_map': demand_map,
        'demand_global': demand_global,
        'demand_q1': demand_q1,
        'demand_q2': demand_q2,
        'pace_map': pace_map,
        'pace_global': pace_global,
        'pace_q1': pace_q1,
        'pace_q2': pace_q2,
    }


def apply_history_features(frame: pd.DataFrame, maps: dict[str, Any]) -> pd.DataFrame:
    out = frame.copy()

    def est_time(row: pd.Series) -> float:
        k = row['route_hour_key']
        if k in maps['route_hour_map']:
            return float(maps['route_hour_map'][k])
        r = row['route_key']
        if r in maps['route_map']:
            return float(maps['route_map'][r])
        return float(maps['global_trip_median'])

    out['estimated_travel_time_min'] = out.apply(est_time, axis=1)

    def demand_idx(row: pd.Series) -> float:
        key = (row['pickup_zone'], int(row['pickup_hour']))
        return float(maps['demand_map'].get(key, maps['demand_global']))

    out['demand_index'] = out.apply(demand_idx, axis=1)
    out['demand_level'] = np.select(
        [
            out['demand_index'] <= maps['demand_q1'],
            out['demand_index'] <= maps['demand_q2'],
        ],
        ['low', 'medium'],
        default='high',
    )

    def traffic_pace(row: pd.Series) -> float:
        key = (row['pickup_borough'], int(row['pickup_hour']))
        return float(maps['pace_map'].get(key, maps['pace_global']))

    out['traffic_pace_min_per_mile'] = out.apply(traffic_pace, axis=1)
    out['traffic_condition'] = np.select(
        [
            out['traffic_pace_min_per_mile'] <= maps['pace_q1'],
            out['traffic_pace_min_per_mile'] <= maps['pace_q2'],
        ],
        ['light', 'moderate'],
        default='heavy',
    )

    # Distance-time interaction for pricing nonlinearities.
    out['distance_x_est_time'] = out['distance'] * out['estimated_travel_time_min']

    return out


train_base = add_base_time_features(train_df)
val_base = add_base_time_features(val_df)
test_base = add_base_time_features(test_df)

history_maps = fit_history_maps(train_base)

train_feat = apply_history_features(train_base, history_maps)
val_feat = apply_history_features(val_base, history_maps)
test_feat = apply_history_features(test_base, history_maps)

train_feat[['trip_id', 'distance', 'pickup_hour', 'route_key', 'estimated_travel_time_min', 'demand_level', 'traffic_condition', 'fare']].head()

## 6) Modeling setup: baseline + TabFM

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true).astype(float)
    y_pred = np.asarray(y_pred).astype(float)

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))

    denom = np.maximum(np.abs(y_true), 1.0)
    mape = float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)
    med_ape = float(np.median(np.abs((y_true - y_pred) / denom)) * 100.0)

    return {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mape_pct': mape,
        'median_ape_pct': med_ape,
    }


def make_model_xy(frame: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, pd.DataFrame]:
    y = pd.to_numeric(frame[TARGET_COL], errors='coerce').to_numpy(dtype=float)
    meta = frame[[c for c in META_COLS if c in frame.columns]].reset_index(drop=True)

    X = frame.drop(columns=[TARGET_COL]).copy()

    # Remove post-trip timestamps/actuals and id.
    for c in ['trip_id', 'pickup', 'dropoff', 'trip_duration_min', 'route_key', 'route_hour_key']:
        if c in X.columns:
            X = X.drop(columns=[c])

    X = X.where(pd.notna(X), np.nan)

    non_numeric_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    for c in non_numeric_cols:
        X[c] = X[c].astype(object)

    return X, y, meta


X_train, y_train, train_meta = make_model_xy(train_feat)
X_val, y_val, val_meta = make_model_xy(val_feat)
X_test, y_test, test_meta = make_model_xy(test_feat)

num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]
cat_cols = [c for c in X_train.columns if c not in num_cols]

logger.info('Features total={} num={} cat={}', len(X_train.columns), len(num_cols), len(cat_cols))


def train_xgboost_baseline(X: pd.DataFrame, y: np.ndarray) -> Pipeline:
    preprocess = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore')),
            ]), cat_cols),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )

    model = XGBRegressor(
        n_estimators=260,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=2,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',
        objective='reg:squarederror',
    )

    pipe = Pipeline([
        ('preprocess', preprocess),
        ('model', model),
    ])
    pipe.fit(X, y)
    return pipe


def get_regression_preds(model: Any, X: pd.DataFrame) -> np.ndarray:
    return np.asarray(model.predict(X)).astype(float)


t0 = time.time()
xgb_model = train_xgboost_baseline(X_train, y_train)
logger.info('XGBoost trained in {:.1f}s', time.time() - t0)

## 7) TabFM regression loading and training

In [ ]:
def _candidate_tabfm_paths() -> list[Path]:
    candidates: list[Path] = []

    if TABFM_CKPT_PATH is not None:
        candidates.append(TABFM_CKPT_PATH)

    # Hugging Face local cache fallback candidates.
    hf_root = Path.home() / '.cache' / 'huggingface' / 'hub' / 'models--google--tabfm-1.0.0-pytorch' / 'snapshots'
    if hf_root.exists():
        for snap in sorted(hf_root.iterdir(), reverse=True):
            if not snap.is_dir():
                continue
            if (snap / 'regression' / 'pytorch_model.bin').exists():
                candidates.append(snap)

    # De-duplicate while preserving order.
    deduped: list[Path] = []
    seen = set()
    for c in candidates:
        key = str(c)
        if key not in seen:
            deduped.append(c)
            seen.add(key)
    return deduped


def load_tabfm_backbone(device: str) -> Any:
    errors: list[str] = []

    for ckpt in _candidate_tabfm_paths():
        try:
            logger.info('Trying TabFM regression checkpoint at {}', ckpt)
            return tabfm_v1_0_0.load(
                model_type='regression',
                checkpoint_path=str(ckpt),
                device=device,
            )
        except Exception as exc:
            errors.append(f'{ckpt}: {exc}')
            logger.warning('Failed checkpoint {} -> {}', ckpt, exc)

    # Final fallback: allow tabfm loader to resolve/download default cache.
    try:
        logger.info('Trying TabFM default regression cache/download resolution')
        return tabfm_v1_0_0.load(
            model_type='regression',
            checkpoint_path=None,
            device=device,
        )
    except Exception as exc:
        errors.append(f'default: {exc}')
        raise RuntimeError('Unable to load TabFM regression weights. Tried: ' + ' | '.join(errors)) from exc


def pick_tabfm_device(requested: str) -> str:
    if requested.startswith('cuda') and torch.cuda.is_available():
        total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info('Detected GPU memory {:.2f} GiB', total_mem_gb)
        if total_mem_gb < 12:
            logger.warning('GPU memory <12 GiB; forcing CPU for TabFM stability')
            return 'cpu'
    return requested


def fit_tabfm_variants(X: pd.DataFrame, y: np.ndarray, requested_device: str) -> dict[str, TabFMRegressor]:
    device = pick_tabfm_device(requested_device)

    if len(X) > TABFM_CONTEXT_MAX_ROWS:
        # Sample context rows uniformly for speed; dataset is time-sorted upstream, so use tail to keep recent context.
        X_fit = X.tail(TABFM_CONTEXT_MAX_ROWS).reset_index(drop=True)
        y_fit = y[-TABFM_CONTEXT_MAX_ROWS:]
        logger.warning('TabFM context capped: {} -> {} rows', len(X), len(X_fit))
    else:
        X_fit, y_fit = X, y

    batch_size = 1 if device == 'cpu' else 2

    if TABFM_FAST_MODE:
        logger.warning('TABFM_FAST_MODE=1 enabled; reduced estimator complexity active')
        default_estimators = 2
        ensemble_estimators = 2
        advanced_estimators = 2
        advanced_norm_methods = ['none']
        advanced_n_feature_crosses = 0
        advanced_n_svd_features = 0
        advanced_total_svd_pool = 32
        advanced_enable_nnls = False
    else:
        default_estimators = 4
        ensemble_estimators = 6
        advanced_estimators = 6
        advanced_norm_methods = ['none', 'power', 'quantile_rtdl']
        advanced_n_feature_crosses = 'sqrt'
        advanced_n_svd_features = 'sqrt'
        advanced_total_svd_pool = 128
        advanced_enable_nnls = True

    models: dict[str, TabFMRegressor] = {}

    m_default = TabFMRegressor(
        model=load_tabfm_backbone(device),
        n_estimators=default_estimators,
        batch_size=batch_size,
        random_state=SEED,
        n_feature_crosses=0,
        n_svd_features=0,
        enable_nnls=False,
        verbose=False,
        num_folds_for_cv=2,
        min_rows_for_single_val_split=100,
    )
    m_default.fit(X_fit, y_fit)
    models['tabfm_default'] = m_default

    m_ensemble = TabFMRegressor.ensemble(
        model=load_tabfm_backbone(device),
        n_estimators=ensemble_estimators,
        batch_size=batch_size,
        random_state=SEED,
        num_folds_for_cv=2,
        min_rows_for_single_val_split=100,
        enable_nnls=not TABFM_FAST_MODE,
        verbose=False,
    )
    m_ensemble.fit(X_fit, y_fit)
    models['tabfm_ensemble_preset'] = m_ensemble

    m_advanced = TabFMRegressor(
        model=load_tabfm_backbone(device),
        n_estimators=advanced_estimators,
        norm_methods=advanced_norm_methods,
        feat_shuffle_method='random',
        permute_categorical=True,
        n_feature_crosses=advanced_n_feature_crosses,
        n_svd_features=advanced_n_svd_features,
        total_svd_pool=advanced_total_svd_pool,
        enable_nnls=advanced_enable_nnls,
        random_state=SEED,
        batch_size=batch_size,
        num_folds_for_cv=2,
        min_rows_for_single_val_split=100,
        verbose=False,
    )
    m_advanced.fit(X_fit, y_fit)
    models['tabfm_advanced_custom'] = m_advanced

    logger.info('TabFM variants trained on device={}', device)
    return models


t1 = time.time()
tabfm_models = fit_tabfm_variants(X_train, y_train, requested_device=DEVICE)
logger.info('TabFM training completed in {:.1f}s', time.time() - t1)


## 8) Evaluate models and pick champion

In [ ]:
def build_eval_slice(X: pd.DataFrame, y: np.ndarray, max_rows: int, keep: str = 'head') -> tuple[pd.DataFrame, np.ndarray]:
    if max_rows == 0 or len(X) <= max_rows:
        return X, y
    if keep == 'head':
        return X.head(max_rows).reset_index(drop=True), y[:max_rows]
    return X.tail(max_rows).reset_index(drop=True), y[-max_rows:]


X_val_eval, y_val_eval = build_eval_slice(X_val, y_val, TABFM_EVAL_MAX_ROWS, keep='head')
X_test_eval, y_test_eval = build_eval_slice(X_test, y_test, TABFM_EVAL_MAX_ROWS, keep='head')
logger.info('Evaluation rows used (val/test): {}/{}', len(X_val_eval), len(X_test_eval))

model_registry: dict[str, Any] = {'xgboost_baseline': xgb_model, **tabfm_models}
rows: list[dict[str, Any]] = []
predictions: dict[str, dict[str, np.ndarray]] = {}

for model_name, model in model_registry.items():
    val_pred = get_regression_preds(model, X_val_eval)
    test_pred = get_regression_preds(model, X_test_eval)
    predictions[model_name] = {'val': val_pred, 'test': test_pred}

    rows.append({'model': model_name, 'split': 'val', **regression_metrics(y_val_eval, val_pred)})
    rows.append({'model': model_name, 'split': 'test', **regression_metrics(y_test_eval, test_pred)})

metrics_df = pd.DataFrame(rows)
val_rank = metrics_df[metrics_df['split'] == 'val'].sort_values('rmse', ascending=True).reset_index(drop=True)
champion_model_name = val_rank.loc[0, 'model']
logger.info('Champion model by validation RMSE: {}', champion_model_name)

champion_model = model_registry[champion_model_name]
champion_val_pred = get_regression_preds(champion_model, X_val)
champion_test_pred = get_regression_preds(champion_model, X_test)
logger.info('Champion full prediction rows (val/test): {}/{}', len(champion_val_pred), len(champion_test_pred))

metrics_df.sort_values(['split', 'rmse'], ascending=[True, True]).reset_index(drop=True)

In [ ]:
plot_df = metrics_df[metrics_df['split'] == 'test'].sort_values('rmse', ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=plot_df, x='rmse', y='model', ax=axes[0], palette='viridis')
axes[0].set_title('Test RMSE')
axes[0].set_xlabel('RMSE')
axes[0].set_ylabel('')

sns.barplot(data=plot_df, x='r2', y='model', ax=axes[1], palette='mako')
axes[1].set_title('Test R2')
axes[1].set_xlabel('R2')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
pred_vs_actual = pd.DataFrame({'actual': y_test, 'predicted': champion_test_pred})

plt.figure(figsize=(6, 6))
plt.scatter(pred_vs_actual['actual'], pred_vs_actual['predicted'], alpha=0.5)
low = min(pred_vs_actual['actual'].min(), pred_vs_actual['predicted'].min())
high = max(pred_vs_actual['actual'].max(), pred_vs_actual['predicted'].max())
plt.plot([low, high], [low, high], '--', color='red')
plt.title(f'Predicted vs Actual Fare ({champion_model_name})')
plt.xlabel('Actual Fare')
plt.ylabel('Predicted Fare')
plt.tight_layout()
plt.show()

## 9) Pricing confidence and quote policy

We estimate fare confidence intervals using validation APE quantiles and model disagreement.
Then assign quote automation tiers:
- `auto_quote`
- `monitor_quote`
- `manual_review_quote`

In [ ]:
val_ape = np.abs((y_val - champion_val_pred) / np.maximum(np.abs(y_val), 1.0))
q50, q80, q90 = np.quantile(val_ape, [0.50, 0.80, 0.90])
logger.info('Validation APE quantiles q50={:.3f} q80={:.3f} q90={:.3f}', q50, q80, q90)

full_test_preds_all = {}
for name, model in model_registry.items():
    full_test_preds_all[name] = get_regression_preds(model, X_test)

pred_matrix = np.vstack([full_test_preds_all[k] for k in sorted(full_test_preds_all.keys())])
disagreement_pct = np.std(pred_matrix, axis=0) / np.maximum(np.abs(np.mean(pred_matrix, axis=0)), 1.0)

test_quotes = test_meta.copy()
test_quotes['actual_fare'] = y_test
test_quotes['predicted_fare'] = champion_test_pred
test_quotes['abs_error'] = np.abs(test_quotes['actual_fare'] - test_quotes['predicted_fare'])
test_quotes['abs_pct_error'] = np.abs((test_quotes['actual_fare'] - test_quotes['predicted_fare']) / np.maximum(np.abs(test_quotes['actual_fare']), 1.0))

test_quotes['fare_lower_q80'] = test_quotes['predicted_fare'] * (1.0 - q80)
test_quotes['fare_upper_q80'] = test_quotes['predicted_fare'] * (1.0 + q80)
test_quotes['model_disagreement_pct'] = disagreement_pct

d1 = float(np.quantile(disagreement_pct, 0.50))
d2 = float(np.quantile(disagreement_pct, 0.85))

test_quotes['quote_tier'] = np.select(
    [
        test_quotes['model_disagreement_pct'] <= d1,
        test_quotes['model_disagreement_pct'] <= d2,
    ],
    ['auto_quote', 'monitor_quote'],
    default='manual_review_quote',
)

coverage_q80 = float(
    ((test_quotes['actual_fare'] >= test_quotes['fare_lower_q80'])
     & (test_quotes['actual_fare'] <= test_quotes['fare_upper_q80'])).mean()
)
logger.info('Empirical q80 interval coverage on test: {:.2%}', coverage_q80)

policy_summary = (
    test_quotes.groupby('quote_tier', dropna=False)
    .agg(
        trips=('trip_id', 'count'),
        avg_pred_fare=('predicted_fare', 'mean'),
        mean_abs_pct_error=('abs_pct_error', 'mean'),
        mean_disagreement_pct=('model_disagreement_pct', 'mean'),
    )
    .reset_index()
    .sort_values('trips', ascending=False)
)

policy_summary

In [ ]:
top_uncertain = test_quotes.sort_values('model_disagreement_pct', ascending=False).head(20)
top_uncertain[['trip_id', 'predicted_fare', 'actual_fare', 'abs_pct_error', 'model_disagreement_pct', 'quote_tier']]

## 10) Persist artifacts

In [ ]:
metrics_path = ARTIFACT_DIR / 'problem6_ridefare_model_metrics.csv'
pred_path = ARTIFACT_DIR / 'problem6_ridefare_predictions_test.parquet'
policy_path = ARTIFACT_DIR / 'problem6_ridefare_quote_actions.csv'
policy_summary_path = ARTIFACT_DIR / 'problem6_ridefare_policy_summary.csv'
runtime_meta_path = ARTIFACT_DIR / 'problem6_ridefare_runtime_meta.json'

metrics_df.to_csv(metrics_path, index=False)
test_quotes.to_csv(policy_path, index=False)
policy_summary.to_csv(policy_summary_path, index=False)

pred_frame = test_quotes[[
    'trip_id',
    'actual_fare',
    'predicted_fare',
    'fare_lower_q80',
    'fare_upper_q80',
    'abs_pct_error',
    'model_disagreement_pct',
    'quote_tier',
]].copy()

pred_cast = pred_frame.copy()
pred_cast['trip_id'] = pred_cast['trip_id'].astype('int64')
for c in ['actual_fare', 'predicted_fare', 'fare_lower_q80', 'fare_upper_q80', 'abs_pct_error', 'model_disagreement_pct']:
    pred_cast[c] = pred_cast[c].astype('float64')
pred_cast['quote_tier'] = pred_cast['quote_tier'].astype(str)

pl.DataFrame({col: pred_cast[col].tolist() for col in pred_cast.columns}).write_parquet(pred_path)

runtime_meta = {
    'seed': SEED,
    'data_url': DATA_URL,
    'sample_train_rows': int(SAMPLE_TRAIN_ROWS),
    'sample_eval_rows': int(SAMPLE_EVAL_ROWS),
    'tabfm_context_max_rows': int(TABFM_CONTEXT_MAX_ROWS),
    'tabfm_eval_max_rows': int(TABFM_EVAL_MAX_ROWS),
    'tabfm_fast_mode': bool(TABFM_FAST_MODE),
    'tabfm_device_preference': TABFM_DEVICE_PREF,
    'tabfm_device_effective': DEVICE,
    'tabfm_checkpoint_override': str(TABFM_CKPT_PATH) if TABFM_CKPT_PATH else None,
    'champion_model': champion_model_name,
    'val_ape_quantiles': {'q50': float(q50), 'q80': float(q80), 'q90': float(q90)},
    'q80_interval_coverage_test': coverage_q80,
}
runtime_meta_path.write_text(json.dumps(runtime_meta, indent=2))

for p in [metrics_path, pred_path, policy_path, policy_summary_path, runtime_meta_path]:
    logger.info('Wrote {}', p)

sorted(p.name for p in ARTIFACT_DIR.glob('problem6_ridefare_*'))

## 11) Deployment notes

Recommended next steps:
1. Enrich with live map ETA and traffic APIs for real pre-trip time estimates.
2. Add geo-embedding features from coordinates/road network.
3. Calibrate uncertainty and quote tiers by route segment.
4. Monitor drift in demand/traffic pace and retrain periodically.